# 2. Original training-data and PCA contract

The original full CSV is **not** `mutant_df.csv` and does not store precomputed
PCA columns. It contains 1,003 records (329 positive, 674 negative), one saved
index, 2,775 raw features and three label/identifier columns. Its raw sample
feature blocks are shown below. The processed pickle is a seven-item list:
`[features, labels, fidelity, PI_flags, BOS_basis, residue_basis, dimensions]`.

| Block | Raw width | Width after ESM PCA |
|---|---:|---:|
| Mutant BOS ESM | 1280 | 50 |
| Mutant residue ESM | 1280 | 50 |
| Kinase-family indicators | 8 | 8 |
| Basic sequence/physicochemical | 59 | 59 |
| MD | 136 | 136 |
| Docking | 12 | 12 |

The original matrix has complete raw ESM vectors for **all 1,003 records**.
Do not delete its source out-of-KD or identity records, regenerate missing
historical values, or replace its embeddings with notebook 1 outputs.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from workflow import (load_training_contract, map_original_training_ids,
                      fit_reference_pcas, assemble_training_features, sha256)

TRAINING_CSV = Path(os.environ.get('KINOME_TRAINING_CSV', '../data/data.csv')).resolve()
DATASET_PKL = Path(os.environ.get('KINOME_DATASET_PKL', '../processed_data/dataset.pkl')).resolve()
SOURCE_DIR = Path(os.environ.get('KINOME_SOURCE_DIR', '.')).resolve()
OUTPUT_DIR = Path(os.environ.get('KINOME_OUTPUT_DIR', 'outputs')).resolve()
TRUST_PICKLE = os.environ.get('KINOME_TRUST_PICKLE', '0') == '1'
RUN_PCA = os.environ.get('KINOME_RUN_PCA', '0') == '1'
HAVE_TRAINING_DATA = TRAINING_CSV.is_file() and DATASET_PKL.is_file() and TRUST_PICKLE
training_hashes = {}
if not HAVE_TRAINING_DATA:
    print('Supply the original full CSV and trusted processed pickle. Set KINOME_TRUST_PICKLE=1.')
    print('No training data are reconstructed or downloaded by this notebook.')

## Validate the actual supplied training files

Only load a pickle you trust: pickle deserialization may execute code.
The checker compares all stored features (including missing structural values)
and labels without modifying either file. The reference bases must be the
supplied 196 x 1280 BOS and 53,664 x 1280 residue matrices—not a newly generated
WT basis. Fidelity is separate metadata, not a family indicator or a sample
weight in the shown original trainer paths. PI availability is 215 with / 788
without structural features, not the activation label.

In [ ]:
if HAVE_TRAINING_DATA:
    training_hashes = {TRAINING_CSV: sha256(TRAINING_CSV), DATASET_PKL: sha256(DATASET_PKL)}
    training, package = load_training_contract(TRAINING_CSV, DATASET_PKL)
    features, labels, fidelity, PI_flags, BOS_basis, residue_basis, dimensions = package
    print({'features': features.shape, 'dimensions': dimensions,
           'positive': int(labels.sum()), 'negative': int((labels == 0).sum()),
           'PI_available': int(PI_flags.sum()),
           'BOS_basis': BOS_basis.shape, 'residue_basis': residue_basis.shape})
    source = pd.read_csv(SOURCE_DIR / 'mutant_df.csv')
    coordinate_map = map_original_training_ids(training, source)
    print('All source IDs and experimental labels match:', len(coordinate_map))

## Legacy IDs and source coordinates

The explicit mapping handles BRAF-NEW/BRAF-OLD -> BRAF, MEK1 -> MAP2K1,
ERBB2 -> HER2, and the known **+24 EGFR source-position offset**. This is only
the original CSV-to-source mapping; do not apply it indiscriminately to new
EGFR cases. The mapping does not rename stored training IDs or change labels.

## Reference PCA: 50 + 50, no raw-embedding standardization

Both original trainers use `PCA(n_components=50, random_state=18)` on their
separate supplied reference bases. They do **not** fit PCA on the 1,003 mutant
sample matrix or on an external test cohort, and do not standardize raw ESM
dimensions first. They fit reference PCA before the cross-validation loop,
not separately in each fold. Numerical results can depend on the historical
library versions; existing inference must reuse its saved model-associated
PCA transforms, not these newly fitted demonstration objects.

The following optional cell validates the trainer configuration. It does not
train a classifier or replace any frozen artifacts.

In [ ]:
if HAVE_TRAINING_DATA and RUN_PCA:
    from threadpoolctl import threadpool_limits
    with threadpool_limits(limits=4):
        pca_first, pca_residue = fit_reference_pcas(BOS_basis, residue_basis)
        X = assemble_training_features(features, pca_first, pca_residue)
    assert X.shape == (1003, 315)
    classical_no_PI = X[:, :167]
    released_NN_main = X[:, :100]
    original_NN_main = X[:, :167]
    original_NN_joint = np.hstack([X[:, 167:], X[:, :100]])
    assert original_NN_joint.shape == (1003, 248)
    print('PCA shapes:', pca_first.components_.shape, pca_residue.components_.shape)
    print('Classical:', classical_no_PI.shape, 'released NN:', released_NN_main.shape,
          'original NN main:', original_NN_main.shape, 'original NN joint:', original_NN_joint.shape)
else:
    print('Reference PCA fitting is disabled or authoritative inputs are unavailable.')

## Model-specific training and inference order

- **Classical no-PI:** concatenate 100 ESM PCs + 8 family + 59 basic = **167**.
  During CV, fit any enabled imputer and then `StandardScaler` on the training
  fold only. Inference uses their saved `transform` methods after saved PCA.
- **Original NN trainer:** the current `use_basic=False` branch actually gives
  a **167**-feature main input; the name is counterintuitive. The joint branch
  has 148 structural + 100 ESM features. `NNtrainer.py` is unscaled; the local
  `NNtrainerScaled.py` variant scales after feature assembly, not before PCA.
- **Released collaborator checkpoints used by current GUI inference:** inspect
  `sub_shared_dim`; the released models have **100**, so they consume ESM PCs
  only. Their frozen PCA pair must accompany those weights.

The original trainer scripts use repeated five-fold stratified CV (five
repeats). Final full-data fitting is separate; its training predictions are
not OOF performance. The original 1,003 report uses the selected supplied OOF
probabilities, not a fresh evaluation of full-data-fitted weights.

### Saved-artifact inference pattern (illustration; not executed)

```python
raw = features_for_new_valid_cases  # explicit original feature-block order
X = assemble_training_features(raw, model_pca_first, model_pca_residue)
X_classical = X[:, :167]
if saved_imputer is not None:
    X_classical = saved_imputer.transform(X_classical)
if saved_scaler is not None:
    X_classical = saved_scaler.transform(X_classical)
# classifier.predict_proba(X_classical)[:, 1]
# Released NN: model-specific saved PCA -> X[:, :checkpoint_sub_shared_dim]
```

Never refit preprocessing on inference cases. Do not assume PCA artifacts
from different model releases are interchangeable. Raw/model probabilities,
GT-hotspot overrides and annotation rules belong to downstream evaluation,
not embedding generation or experimental-label construction.

In [ ]:
assert all(sha256(path) == expected for path, expected in training_hashes.items())
print('Original training CSV and processed pickle unchanged; no classifier was trained.')